# HAR Preprocessing Notebook
This notebook performs signal standardization, temporal windowing, Butterworth filtering, feature extraction, normalization, and exports scaling parameters for the UCI HAR dataset.

The reason we do the preprocessing in this step is because raw data from the train.csv and test.csv datasets is noisy and high-dimensional. This means the ML cannot learn from it directly. Trash in = trash out. We have to transform them into meaningful features.

In [3]:
import numpy as np
import pandas as pd
from scipy.signal import butter, lfilter
import json

In [4]:
# load csv files
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [5]:
# explore dfs 
print('Train DataFrame shape:', train_df.shape)
print('Test DataFrame shape:', test_df.shape)
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
train_df.head(), test_df.head()

Train DataFrame shape: (7352, 563)
Test DataFrame shape: (2947, 563)
Train columns: ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z', 'tBodyAcc-max()-X', 'tBodyAcc-max()-Y', 'tBodyAcc-max()-Z', 'tBodyAcc-min()-X', 'tBodyAcc-min()-Y', 'tBodyAcc-min()-Z', 'tBodyAcc-sma()', 'tBodyAcc-energy()-X', 'tBodyAcc-energy()-Y', 'tBodyAcc-energy()-Z', 'tBodyAcc-iqr()-X', 'tBodyAcc-iqr()-Y', 'tBodyAcc-iqr()-Z', 'tBodyAcc-entropy()-X', 'tBodyAcc-entropy()-Y', 'tBodyAcc-entropy()-Z', 'tBodyAcc-arCoeff()-X,1', 'tBodyAcc-arCoeff()-X,2', 'tBodyAcc-arCoeff()-X,3', 'tBodyAcc-arCoeff()-X,4', 'tBodyAcc-arCoeff()-Y,1', 'tBodyAcc-arCoeff()-Y,2', 'tBodyAcc-arCoeff()-Y,3', 'tBodyAcc-arCoeff()-Y,4', 'tBodyAcc-arCoeff()-Z,1', 'tBodyAcc-arCoeff()-Z,2', 'tBodyAcc-arCoeff()-Z,3', 'tBodyAcc-arCoeff()-Z,4', 'tBodyAcc-correlation()-X,Y', 'tBodyAcc-correlation()-X,Z', 'tBodyAcc-correlation

(   tBodyAcc-mean()-X  tBodyAcc-mean()-Y  tBodyAcc-mean()-Z  tBodyAcc-std()-X  \
 0           0.288585          -0.020294          -0.132905         -0.995279   
 1           0.278419          -0.016411          -0.123520         -0.998245   
 2           0.279653          -0.019467          -0.113462         -0.995380   
 3           0.279174          -0.026201          -0.123283         -0.996091   
 4           0.276629          -0.016570          -0.115362         -0.998139   
 
    tBodyAcc-std()-Y  tBodyAcc-std()-Z  tBodyAcc-mad()-X  tBodyAcc-mad()-Y  \
 0         -0.983111         -0.913526         -0.995112         -0.983185   
 1         -0.975300         -0.960322         -0.998807         -0.974914   
 2         -0.967187         -0.978944         -0.996520         -0.963668   
 3         -0.983403         -0.990675         -0.997099         -0.982750   
 4         -0.980817         -0.990482         -0.998321         -0.979672   
 
    tBodyAcc-mad()-Z  tBodyAcc-max()-X  ..

### Preprocessing

We have to start with standardization to make sure our data is on a similar scale, which is important for most algorithms, especially NNs. It will also remove biases from the data.

In [8]:

from sklearn.preprocessing import MinMaxScaler


# drop non feature columns
feature_columns = [col for col in train_df.columns if col not in ['Activity', 'subject']]
X_train = train_df[feature_columns].values
X_test = test_df[feature_columns].values


# standardiziation
# The scale -1 to 1 is preffered for NNs, it reduces bias, since NNs rely on multiplications, 
# numbers beyond -1 and 1 could cause weighs to grow too fast and cause floating point overflow

scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaler_params = {
    "min": scaler.data_min_.tolist(),
    "scale": scaler.scale_.tolist(),
    "feature_names": feature_columns 
}

with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f)

print('Feature scaling parameters saved to scaler_params.json')



Feature scaling parameters saved to scaler_params.json


In [9]:
# split data into train-test split

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train_scaled, train_df['Activity'].values, test_size=0.2, random_state=116, stratify=train_df['Activity'].values)

print('Train set:', X_train.shape)
print('Validation set:', X_val.shape)

Train set: (5881, 561)
Validation set: (1471, 561)


### Now x_train will be our train set, X_val will be validation set, and X_test_scaled will be the test set.

In [7]:

# we use butterworth low-pass filter to remove high-frequency noise from the sensor data,
# which can improve the performance of the model by focusing on the relevant signal patterns for human activity recognition.

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = lfilter(b, a, data) 
    return y

# Parameters
fs = 50  # Sampling frequency
cutoff = 0.3  # Cutoff frequency
window_size = 128
step_size = window_size // 2



filtered_train = np.apply_along_axis(butter_lowpass_filter, 0, X_train_scaled, cutoff, fs)
filtered_test = np.apply_along_axis(butter_lowpass_filter, 0, X_test_scaled, cutoff, fs)


# Windowing

def create_windows(data, window_size, step_size):
    return np.array([data[i:i+window_size] for i in range(0, data.shape[0] - window_size + 1, step_size)])

train_windows = create_windows(filtered_train, window_size, step_size)
test_windows = create_windows(filtered_test, window_size, step_size)

def assign_window_labels(labels, window_size, step_size):
    return [labels[i:i+window_size].mode()[0] for i in range(0, len(labels) - window_size + 1, step_size)]

train_labels = train_df['Activity']
test_labels = test_df['Activity']
train_window_labels = assign_window_labels(train_labels, window_size, step_size)
test_window_labels = assign_window_labels(test_labels, window_size, step_size)

In [10]:
# --- Rational Agent MLP: Universal Function Approximator ---


# Step A: Layer Initialization (He Initialization)
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        # He initialization for weights
        self.weights = np.random.randn(n_inputs, n_neurons) * np.sqrt(2. / n_inputs)
        self.biases = np.zeros((1, n_neurons))
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases

# Step B: Non-linear Activation (ReLU)
class Activation_ReLU:
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)

# Step C: Output Layer (Softmax)
class Activation_Softmax:
    def forward(self, inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

# Cross-entropy loss with epsilon clipping for numerical stability
class Loss_CategoricalCrossentropy:
    def forward(self, y_pred, y_true):
        # Clip predictions to prevent log(0)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
        # If labels are one-hot encoded, turn them into class indices
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)
        correct_confidences = y_pred_clipped[range(len(y_pred)), y_true]
        negative_log_likelihoods = -np.log(correct_confidences)
        return np.mean(negative_log_likelihoods)

# Example: Model assembly (modular, object-oriented)
class Model:
    def __init__(self):
        self.layers = []
    def add(self, layer):
        self.layers.append(layer)
    def forward(self, X):
        output = X
        for layer in self.layers:
            layer.forward(output)
            output = layer.output
        return output

# Example usage:
model = Model()
model.add(Layer_Dense(561, 64))
model.add(Activation_ReLU())
model.add(Layer_Dense(64, 64))
model.add(Activation_ReLU())
model.add(Layer_Dense(64, 6))
model.add(Activation_Softmax())

# Forward pass (replace X_train_part with your actual data)
# output = model.forward(X_train_part)

# Example scaling_params.json format for React:
# {
#   "min": [min1, min2, ..., min561],
#   "scale": [scale1, scale2, ..., scale561],
#   "feature_names": ["f1", "f2", ..., "f561"]
# }
